# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [1]:
import numpy as np 
import pandas as pd

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use. 

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text. 

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available. 

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [2]:
# news = pd.read_csv('/kaggle/input/topic-labeled-news-dataset/labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT="title"
# TOPIC="topic"

#news = pd.read_csv('/kaggle/input/bbc-news/bbc_news.csv')
#MAX_NEWS = 1000
#DOCUMENT="description"
#TOPIC="title"

#news = pd.read_csv('/kaggle/input/mit-ai-news-published-till-2023/articles.csv')
#MAX_NEWS = 100
#DOCUMENT="Article Body"
#TOPIC="Article Header"

news = "PICK A DATASET" #Ideally pick one from the commented ones above

ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [7]:
from datasets import load_dataset

dataset = load_dataset("ag_news")
news = dataset["train"].to_pandas()

DOCUMENT = "text"
TOPIC = "label"
MAX_NEWS = 1000

c:\Users\macat\anaconda3\envs\langchain-mq\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\macat\anaconda3\envs\langchain-mq\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\macat\.cache\huggingface\hub\datasets--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate dev

In [8]:
news["id"] = news.index
news.head()

,text,label,id
0,Wall St. Bears Claw Back Into the Black (Reute...,2,0
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2,1
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2,3
4,"Oil prices soar to all-time record, posing new...",2,4


In [9]:
#Because it is just a course we select a small portion of News.
subset_news = news.head(MAX_NEWS)

## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database. 

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior. 

In [10]:
!pip install chromadb

  Using cached protobuf-6.33.1-cp310-abi3-win_amd64.whl.metadata (593 bytes)
Using cached protobuf-6.33.1-cp310-abi3-win_amd64.whl (436 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3


In [11]:
import chromadb
from chromadb.config import Settings

Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters 
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance. 

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that. 

In [12]:
chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it. 

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [13]:
collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)
    

It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***. 
* In the **document** we store the big text, it's a different column in each Dataset. 
* In **metadatas**, we can informa a list of topics. 
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS. 


In [14]:

collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
)

In [15]:
results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

{'ids': [['id51', 'id320', 'id846', 'id311', 'id957', 'id714', 'id318', 'id845', 'id326', 'id390']], 'embeddings': None, 'documents': [["Delightful Dell The company's results show that it's not grim all over tech world. Just all of it that isn't Dell.", "Lenovo revenue grows, but problems persist China's largest PC maker sees surge in profits but loses market share to multinational companies.", "CompUSA Stores to Sell Gateway PCs  NEW YORK (Reuters) - Gateway Inc. &lt;GTW.N&gt;, the computer  maker that recently shuttered its chain of 188 stores, will  begin selling its desktops at CompUSA Inc.'s outlets later this  week, the two companies said on Monday.", "Briefly: HP partners with 3G gear maker roundup Plus: AMD starts shipping 90-nanometer chips...ABC.com to air on Real's SuperPass...Lenovo revenue grows, but problems persist.", "'SP2' a Must For XP Users (washingtonpost.com) washingtonpost.com - To get an idea of how Windows got to be such a mess, think of it as a house that was b

## Vector MAP

In [16]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [17]:

getado = collection.get(ids="id141", 
                       include=["documents", "embeddings"])


In [18]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
word_vectors

array([[-1.27782673e-01,  2.36220434e-02, -3.06040682e-02,
        -2.80056130e-02,  6.57772571e-02,  4.11881879e-02,
        -7.35446345e-03, -3.43470764e-03,  4.47358228e-02,
         1.58321466e-02,  9.49877873e-02,  1.57126054e-01,
        -2.67630033e-02,  1.70217697e-02, -4.34782989e-02,
         3.44269834e-02, -8.61885864e-03, -2.37583816e-02,
         5.28481938e-02, -4.74139675e-02, -6.66537285e-02,
        -3.82137373e-02,  1.08763546e-01, -3.47380713e-02,
         1.64689142e-02,  5.99439144e-02, -1.76914092e-02,
         9.91218444e-03, -3.37578240e-03, -4.11550403e-02,
        -1.48818288e-02,  2.70390734e-02, -2.02636383e-02,
        -2.26624236e-02,  6.74024830e-03, -5.02553172e-02,
         3.66574936e-02,  8.18102956e-02,  6.12395816e-02,
         1.57647661e-03,  1.18321888e-01, -4.67841737e-02,
        -7.09753530e-03,  1.28431708e-01, -7.01322481e-02,
        -5.41456975e-04, -4.06708457e-02,  1.12904169e-01,
        -6.77321479e-02, -3.16088535e-02,  1.02144293e-0

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents. 

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search. 


## Loading the model and creating the prompt
TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models. 

We are importing: 
* **Autotokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.
* **AutoModelForCasualLLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models), or the model used in this notebook ***databricks/dolly-v2-3b***.
* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks, such as text generation (our case) or text classification. 

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It have 3billion paramaters, more than enough for our sample, and works much better than GPT2. 

Please, feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending), you need to search for NLP models trained for text-generation. My recomendation is choose "small" models, or we will run out of memory in kaggle.  


In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "databricks/dolly-v2-3b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)



c:\Users\macat\anaconda3\envs\langchain-mq\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\macat\.cache\huggingface\hub\models--databricks--dolly-v2-3b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


The next step is to initialize the pipeline using the objects created above. 

The model's response is limited to 256 tokens, for this project I'm not interested in a longer response, but it can easily be extended to whatever length you want.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU for processing the text generation.  

In [20]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device_map="auto",
)

Device set to use cpu


## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user. 

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**. 

We only need to join the two parts together to create the prompt that we are going to send to the model. 

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part. 

In [21]:
question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
prompt_template

"Relevant context: #Delightful Dell The company's results show that it's not grim all over tech world. Just all of it that isn't Dell. #Lenovo revenue grows, but problems persist China's largest PC maker sees surge in profits but loses market share to multinational companies. #CompUSA Stores to Sell Gateway PCs  NEW YORK (Reuters) - Gateway Inc. &lt;GTW.N&gt;, the computer  maker that recently shuttered its chain of 188 stores, will  begin selling its desktops at CompUSA Inc.'s outlets later this  week, the two companies said on Monday. #Briefly: HP partners with 3G gear maker roundup Plus: AMD starts shipping 90-nanometer chips...ABC.com to air on Real's SuperPass...Lenovo revenue grows, but problems persist. #'SP2' a Must For XP Users (washingtonpost.com) washingtonpost.com - To get an idea of how Windows got to be such a mess, think of it as a house that was built on an island in the middle of a lake, deep in the countryside. #Gateway lands on another retailer's shelves Company is s

Now all that remains is to send the prompt to the model and wait for its response!


In [ ]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])